# PASO 1 — Análisis Exploratorio de Datos (EDA)
## Proyecto: Análisis de Churn en TELCO

**Equipo:** Data & BI

**Objetivo de esta fase:** entender la estructura, calidad y distribución de los 5
CSV de origen (`demographics`, `location`, `population`, `services`, `status`) **antes**
de limpiar, integrar o modelar nada.

Para cada tabla analizamos:
- **Estructura básica**: filas, columnas, clave primaria, tipos de dato.
- **Calidad de los datos**: nulos, duplicados, tipos incorrectos.
- **Distribuciones y valores**: estadísticos descriptivos, valores únicos, valores
  imposibles o extremos.

Al final respondemos las **6 preguntas de reflexión** del documento de requisitos.

## 0. Configuración inicial

Cargamos librerías y los 5 datasets en bruto desde `data/raw/`.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None) # Mostrar todas las columnas
pd.set_option('display.width', 250) # Ajustar el ancho de la pantalla para mostrar todas las columnas
pd.set_option('display.float_format', lambda x: f'{x:,.2f}') # Formato de visualización para números flotantes

RAW_PATH = '../data/raw/'

demographics = pd.read_csv(RAW_PATH + 'Telco_customer_churn_demographics.csv')
location     = pd.read_csv(RAW_PATH + 'Telco_customer_churn_location.csv')
population   = pd.read_csv(RAW_PATH + 'Telco_customer_churn_population.csv')
services     = pd.read_csv(RAW_PATH + 'Telco_customer_churn_services.csv')
status       = pd.read_csv(RAW_PATH + 'Telco_customer_churn_status.csv')

# Crear un diccionario con todas las tablas
tablas = {
    'demographics': demographics,
    'location': location,
    'population': population,
    'services': services,
    'status': status,
}

# df.shape[0] -> número de filas
# df.shape[1] -> número de columnas

for nombre, df in tablas.items():
    print(f"{nombre:<15} -> {df.shape[0]} filas x {df.shape[1]} columnas")

demographics    -> 7043 filas x 9 columnas
location        -> 7043 filas x 8 columnas
population      -> 1671 filas x 3 columnas
services        -> 7043 filas x 30 columnas
status          -> 7043 filas x 8 columnas


## 1. Tabla `demographics`

### 1.1 Estructura básica

In [2]:
# shape para ver el número de filas y columnas del dataframe
print("Shape:", demographics.shape)

# head para ver las primeras filas del dataframe
demographics.head()

Shape: (7043, 9)


,Customer ID,Count,Gender,Age,Under 30,Senior Citizen,Married,Dependents,Number of Dependents
0,8779-QRDMV,1,M,78,No,Yes,No,No,0
1,7495-OOKFY,1,F,74,No,Yes,Yes,Yes,1
2,1658-BYGOY,1,M,71,No,Yes,No,Yes,3
3,4598-XLKNJ,1,F,78,No,Yes,Yes,Yes,1
4,4846-WHAFZ,1,F,80,No,Yes,Yes,Yes,1


In [3]:
# info para ver información general del dataframe
demographics.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   Customer ID           7043 non-null   str  
 1   Count                 7043 non-null   int64
 2   Gender                7043 non-null   str  
 3   Age                   7043 non-null   int64
 4   Under 30              7043 non-null   str  
 5   Senior Citizen        7043 non-null   str  
 6   Married               7043 non-null   str  
 7   Dependents            7043 non-null   str  
 8   Number of Dependents  7043 non-null   int64
dtypes: int64(3), str(6)
memory usage: 495.3 KB


In [4]:
# Clave primaria: ¿Customer ID es unico?

# len para ver el número de filas del dataframe
print("Filas totales:", len(demographics))

# nunique para ver el número de valores únicos en la columna 'Customer ID'
print("Customer ID unicos:", demographics['Customer ID'].nunique())

# duplicated para ver el número de valores duplicados en la columna 'Customer ID'
print("Customer ID duplicados:", demographics['Customer ID'].duplicated().sum())

Filas totales: 7043
Customer ID unicos: 7043
Customer ID duplicados: 0


**Estructura básica — interpretación**

- **Filas / columnas:** 7.043 filas x 9 columnas.
- **Clave primaria:** sí, `Customer ID` es única (7.043 valores distintos, 0 duplicados).
  No hay ninguna otra columna que identifique unívocamente al cliente.
- **Tipos de columna:**
  - `Customer ID` → identificador (texto, formato `XXXX-XXXXX`).
  - `Count` → numérica, pero en realidad es una **constante = 1** (residuo típico de
    exportaciones de Excel/Power BI, no aporta información).
  - `Gender` → categórica (sexo del cliente).
  - `Age` → numérica (edad en años).
  - `Under 30`, `Senior Citizen`, `Married`, `Dependents` → categóricas binarias (Yes/No),
    las tres primeras son en realidad **flags derivados de `Age`**.
  - `Number of Dependents` → numérica (nº de personas a cargo).

### 1.2 Calidad de los datos

In [5]:
# Nulos: isnull() para ver si hay valores nulos en el dataframe
nulos = demographics.isnull().sum()
print("Valores nulos por columna:")
print(nulos[nulos > 0] if nulos.sum() > 0 else "Sin valores nulos")

# Duplicados
print("\nFilas duplicadas (todas las columnas):", demographics.duplicated().sum())

Valores nulos por columna:
Sin valores nulos

Filas duplicadas (todas las columnas): 0


In [6]:
# ¿Tipos correctos? -> revisamos columnas categoricas

# value_counts() para ver la distribución de valores en las columnas categóricas
for col in ['Gender', 'Under 30', 'Senior Citizen', 'Married', 'Dependents']:
    print(f"\n{col}:")
    print(demographics[col].value_counts(dropna=False))


Gender:
Gender
Male      2918
Female    2850
F          638
M          637
Name: count, dtype: int64

Under 30:
Under 30
No     5642
Yes    1401
Name: count, dtype: int64

Senior Citizen:
Senior Citizen
No     5901
Yes    1142
Name: count, dtype: int64

Married:
Married
No     3641
Yes    3402
Name: count, dtype: int64

Dependents:
Dependents
No     5416
Yes    1627
Name: count, dtype: int64


**Calidad de los datos — interpretación**

- **Nulos:** no hay ningún valor nulo en esta tabla.
- **Duplicados:** no hay filas duplicadas ni `Customer ID` repetidos.
- **Tipos de datos:** todas las columnas numéricas (`Age`, `Number of Dependents`,
  `Count`) ya vienen como `int64`, no hay números almacenados como texto.
- **Inconsistencia de texto detectada en `Gender`:** la columna mezcla dos
  convenciones distintas:
  - `Male` / `Female` (2.918 / 2.850 registros)
  - `M` / `F` (637 / 638 registros)

  Es decir, el **mismo concepto** (sexo) está codificado de dos formas distintas según
  el bloque de filas. Esto es un **error de homogeneización** que hay que corregir en
  la ETL (normalizar todo a `Male`/`Female` o a `M`/`F`), porque si no, cualquier
  `groupby('Gender')` o gráfico contaría 4 categorías en vez de 2.

### 1.3 Distribuciones y valores

In [7]:
# describe() para ver estadísticas descriptivas de las columnas numéricas
demographics[['Age', 'Number of Dependents']].describe()

,Age,Number of Dependents
count,"7,043.00","7,043.00"
mean,47.47,0.47
std,18.39,0.96
min,19.00,0.00
25%,33.00,0.00
50%,46.00,0.00
75%,60.00,0.00
max,119.00,9.00


In [8]:
# ¿Hay edades imposibles / extremas?
print("Clientes con Age > 100:", (demographics['Age'] > 100).sum())
print(demographics[demographics['Age'] > 100]['Age'].describe())

# ¿En que filas estan? -> index para ver el rango de indices con Age > 100
idx_extremos = demographics[demographics['Age'] > 100].index

# min() y max() para ver el rango de indices
print("\nRango de indices con Age > 100:", idx_extremos.min(), "-", idx_extremos.max())

Clientes con Age > 100: 109
count   109.00
mean    109.91
std       5.42
min     101.00
25%     106.00
50%     109.00
75%     115.00
max     119.00
Name: Age, dtype: float64

Rango de indices con Age > 100: 6933 - 7042


In [9]:
# Verificamos coherencia de los flags derivados de Age
print("Rango de Age segun 'Under 30':")

# groupby() para agrupar por la columna 'Under 30' 
# y luego aplicar agg() para obtener min, max y count de Age
print(demographics.groupby('Under 30')['Age'].agg(['min', 'max', 'count']))

print("\nRango de Age segun 'Senior Citizen':")
print(demographics.groupby('Senior Citizen')['Age'].agg(['min', 'max', 'count']))

# Inconsistencias explicitas
inc_under30 = demographics[(demographics['Under 30'] == 'Yes') & (demographics['Age'] >= 30)]
inc_senior  = demographics[(demographics['Senior Citizen'] == 'No') & (demographics['Age'] >= 65)]
print(f"\nFilas con 'Under 30'='Yes' pero Age >= 30: {len(inc_under30)}")
print(f"Filas con 'Senior Citizen'='No' pero Age >= 65: {len(inc_senior)}")

Rango de Age segun 'Under 30':
          min  max  count
Under 30                 
No         30  119   5642
Yes        19  118   1401

Rango de Age segun 'Senior Citizen':
                min  max  count
Senior Citizen                 
No               19  119   5901
Yes              65  119   1142

Filas con 'Under 30'='Yes' pero Age >= 30: 20
Filas con 'Senior Citizen'='No' pero Age >= 65: 89


In [10]:
# Number of Dependents vs Dependents (Yes/No)
demographics.groupby('Dependents')['Number of Dependents'].agg(['min', 'max', 'mean'])

,min,max,mean
Dependents,,,
No,0,0,0.00
Yes,1,9,2.03


**Distribuciones y valores — interpretación**

- **`Age`**: rango teórico 19-119, media ≈ 47,5, mediana = 46. El **máximo de 119 años
  es un valor imposible** para un cliente activo de telecomunicaciones. Al
  investigarlo, encontramos que **109 clientes tienen `Age` > 100**, y todos ellos
  están concentrados en el **bloque final del fichero** (filas 6.933-7.042, las
  últimas ~110 filas). Esto tiene toda la pinta de ser un **bloque de datos
  sintéticos/erróneos inyectado deliberadamente** para el ejercicio, no un problema
  de recogida real distribuido por todo el dataset.
- **Coherencia de los flags derivados:**
  - `Senior Citizen` debería ser `Yes` siempre que `Age >= 65`. Sin embargo, **89 de
    los 109 clientes con `Age > 100` tienen `Senior Citizen = 'No'`**, lo cual es
    contradictorio.
  - `Under 30` debería ser `Yes` solo si `Age < 30`. Encontramos **20 filas** (todas
    dentro del mismo bloque final) con `Under 30 = 'Yes'` pese a tener `Age` entre
    101 y 119.
  - Para el resto del dataset (filas 0-6932) **no hay ninguna inconsistencia**: los
    flags `Under 30` y `Senior Citizen` son perfectamente coherentes con `Age`.
- **`Number of Dependents`**: rango 0-9, media ≈ 0,47, mediana = 0 (la mayoría de
  clientes no tiene personas a cargo). Es totalmente coherente con `Dependents`:
  cuando `Dependents = 'No'`, `Number of Dependents` es siempre 0; cuando
  `Dependents = 'Yes'`, va de 1 a 9.
- **Conclusión:** el problema real de calidad en esta tabla **no son los nulos**
  (no hay ninguno), sino un **bloque final de ~110 registros con edades
  anómalas e inconsistentes con sus propios flags**, y una **inconsistencia de
  formato en `Gender`** que afecta a ~1.275 filas.

## 2. Tabla `location`

### 2.1 Estructura básica

In [11]:
print("Shape:", location.shape)
location.head()

Shape: (7043, 8)


,Customer ID,Count,Country,State,City,Zip Code,Latitude,Longitude
0,8779-QRDMV,1,United States,California,Los Angeles,90022,34.02,-118.16
1,7495-OOKFY,1,United States,California,Los Angeles,90063,34.04,-118.19
2,1658-BYGOY,1,United States,California,Los Angeles,90065,34.11,-118.23
3,4598-XLKNJ,1,United States,California,Inglewood,90303,33.94,-118.33
4,4846-WHAFZ,1,United States,California,Whittier,90602,33.97,-118.02


In [12]:
location.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Customer ID  7043 non-null   str    
 1   Count        7043 non-null   int64  
 2   Country      7043 non-null   str    
 3   State        7043 non-null   str    
 4   City         7043 non-null   str    
 5   Zip Code     7043 non-null   int64  
 6   Latitude     7043 non-null   float64
 7   Longitude    7043 non-null   float64
dtypes: float64(2), int64(2), str(4)
memory usage: 440.3 KB


In [13]:
print("Customer ID unicos:", location['Customer ID'].nunique())
print("Customer ID duplicados:", location['Customer ID'].duplicated().sum())

Customer ID unicos: 7043
Customer ID duplicados: 0


**Estructura básica — interpretación**

- **Filas / columnas:** 7.043 filas x 8 columnas.
- **Clave primaria:** `Customer ID`, única y sin duplicados (igual que en
  `demographics`).
- **Tipos de columna:**
  - `Customer ID` → identificador.
  - `Count` → constante = 1 (igual que en `demographics`, sin información).
  - `Country`, `State` → categóricas, pero **constantes** (todo el dataset es de
    "United States" / "California").
  - `City` → categórica (1.106 ciudades, coincide con el contexto de negocio).
  - `Zip Code` → numérica por tipo de dato, pero **conceptualmente es un código
    identificador geográfico**, no una cantidad (no tiene sentido sumarlo o
    promediarlo).
  - `Latitude`, `Longitude` → numéricas (coordenadas geográficas).

### 2.2 Calidad de los datos

In [14]:
print("Nulos por columna:")

# location.isnull().sum() indica el número de valores nulos por columna
# location.isnull().sum() > 0 filtra las columnas que tienen valores nulos
# if location.isnull().sum().sum() > 0 indica si hay algún valor nulo en todo el dataframe
# else para indicar que no hay valores nulos
print(location.isnull().sum()[location.isnull().sum() > 0] if location.isnull().sum().sum() > 0 else "Sin valores nulos")

print("\nFilas duplicadas:", location.duplicated().sum())

# unique() para ver los valores únicos en las columnas 'Country' y 'State'
print("\nCountry:", location['Country'].unique())
print("State:", location['State'].unique())

Nulos por columna:
Sin valores nulos

Filas duplicadas: 0

Country: <StringArray>
['United States']
Length: 1, dtype: str
State: <StringArray>
['California']
Length: 1, dtype: str


In [15]:
# Inconsistencias de texto en 'City' (mayusculas/espacios)
ciudades = location['City'].unique() # unique() para obtener los valores únicos de la columna 'City'

# Normalizamos las ciudades para detectar posibles duplicados por mayusculas/espacios
# pd.Series para convertir el array de ciudades en una serie de pandas
# str.strip() para eliminar espacios en blanco al inicio y al final de cada ciudad
# str.lower() para convertir todas las ciudades a minúsculas
ciudades_normalizadas = pd.Series(ciudades).str.strip().str.lower()

print("Ciudades unicas:", len(ciudades))
print("Posibles duplicados por mayusculas/espacios:", ciudades_normalizadas.duplicated().sum())

Ciudades unicas: 1106
Posibles duplicados por mayusculas/espacios: 0


**Calidad de los datos — interpretación**

- **Nulos:** no hay valores nulos en ninguna columna.
- **Duplicados:** no hay filas duplicadas.
- **Tipos de datos:** correctos, no hay numéricos almacenados como texto.
- **Consistencia de texto:** `Country` y `State` son constantes (`United States`,
  `California`) — no aportan varianza, pero confirman el alcance geográfico descrito
  en el contexto de negocio. En `City` **no encontramos inconsistencias de
  mayúsculas/espacios** (1.106 ciudades únicas tanto en bruto como normalizadas).

### 2.3 Distribuciones y valores

In [16]:
location[['Zip Code', 'Latitude', 'Longitude']].describe()

,Zip Code,Latitude,Longitude
count,"7,043.00","7,043.00","7,043.00"
mean,"93,486.07",36.20,-119.76
std,"1,856.77",2.47,2.15
min,"90,001.00",32.56,-124.30
25%,"92,101.00",33.99,-121.79
50%,"93,518.00",36.21,-119.60
75%,"95,329.00",38.16,-117.97
max,"96,150.00",41.96,-114.19


In [17]:
print("Numero de ciudades unicas:", location['City'].nunique())
print("\nTop 10 ciudades con mas clientes:")
print(location['City'].value_counts().head(10))

print("\nClientes en San Diego:", (location['City'] == 'San Diego').sum())

Numero de ciudades unicas: 1106

Top 10 ciudades con mas clientes:
City
Los Angeles      293
San Diego        285
San Jose         112
Sacramento       108
San Francisco    104
Fresno            61
Long Beach        60
Oakland           52
Escondido         51
Stockton          44
Name: count, dtype: int64

Clientes en San Diego: 285


**Distribuciones y valores — interpretación**

- **`Zip Code`**: rango 90001-96150, todos códigos postales reales de California.
- **`Latitude` / `Longitude`**: rango 32,56 - 41,96 / -124,30 - -114,19, que coincide
  exactamente con el "bounding box" geográfico del estado de California — **no hay
  coordenadas fuera de rango ni valores imposibles**.
- **`City`**: **1.106 ciudades únicas**, lo cual **coincide exactamente** con el dato
  que nos da Amalia en el contexto de negocio ("presencia en 1.106 ciudades del
  estado de California"). Esto es una buena señal de calidad: el dataset es
  internamente consistente con la narrativa de negocio.
- **San Diego** tiene **285 clientes**, la 2ª ciudad con más clientes después de Los
  Ángeles (293). Esto será relevante para evaluar la hipótesis de Sebastián sobre la
  competencia en San Diego.
- **Conclusión:** esta es, junto con `population`, la tabla **más limpia** de las 5 —
  no se detecta ningún problema de calidad.

## 3. Tabla `population`

### 3.1 Estructura básica

In [18]:
print("Shape:", population.shape)
population.head()

Shape: (1671, 3)


,ID,Zip Code,Population
0,1,90001,54492
1,2,90002,44586
2,3,90003,58198
3,4,90004,67852
4,5,90005,43019


In [19]:
population.info()

<class 'pandas.DataFrame'>
RangeIndex: 1671 entries, 0 to 1670
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   ID          1671 non-null   int64
 1   Zip Code    1671 non-null   int64
 2   Population  1671 non-null   int64
dtypes: int64(3)
memory usage: 39.3 KB


In [20]:
print("Filas:", len(population))
print("Zip Code unicos:", population['Zip Code'].nunique())
print("Zip Code duplicados:", population['Zip Code'].duplicated().sum())

# Verificar si 'ID' es simplemente la posición de la fila + 1 
# .all para verificar si todos los valores cumplen la condición
print("\n'ID' es simplemente la posicion de la fila + 1?",
      (population['ID'] == population.index + 1).all()) 

Filas: 1671
Zip Code unicos: 1671
Zip Code duplicados: 0

'ID' es simplemente la posicion de la fila + 1? True


**Estructura básica — interpretación**

- **Filas / columnas:** 1.671 filas x 3 columnas. Esta tabla **no está a nivel de
  cliente**, sino a **nivel de código postal** (1.671 zip codes distintos de
  California).
- **Clave primaria:** `Zip Code` (único, sin duplicados). **No** es `Customer ID` —
  esta tabla no tiene esa columna.
- **Tipos de columna:**
  - `ID` → numérica, pero es **un índice secuencial 1..1671** (`ID == fila + 1` para
    el 100% de las filas), totalmente redundante.
  - `Zip Code` → identificador geográfico (clave de esta tabla).
  - `Population` → numérica (habitantes del código postal).

### 3.2 Calidad de los datos

In [21]:
print("Nulos por columna:")
print(population.isnull().sum()[population.isnull().sum() > 0] if population.isnull().sum().sum() > 0 else "Sin valores nulos")

print("\nFilas duplicadas:", population.duplicated().sum())

Nulos por columna:
Sin valores nulos

Filas duplicadas: 0


In [22]:
# Relacion con location: ¿todos los zip code de location existen en population?
# set sirve para obtener los valores únicos de una columna y convertirlos en un conjunto
zips_location = set(location['Zip Code'].unique())
zips_population = set(population['Zip Code'].unique())

print("Zip codes distintos en location:", len(zips_location))
print("Zip codes distintos en population:", len(zips_population))

# estos son los zip codes que estan en location pero no en population
print("Zip codes de location SIN match en population:", len(zips_location - zips_population))

# estos son los zip codes que estan en population pero no en location
print("Zip codes de population que ningun cliente usa:", len(zips_population - zips_location))

Zip codes distintos en location: 1626
Zip codes distintos en population: 1671
Zip codes de location SIN match en population: 0
Zip codes de population que ningun cliente usa: 45


**Calidad de los datos — interpretación**

- **Nulos:** ninguno.
- **Duplicados:** ninguno (`Zip Code` único en las 1.671 filas).
- **Tipos de datos:** correctos.
- **Relación con `location`:** los **1.626 zip codes** que aparecen en `location`
  están **todos** presentes en `population` (0 sin match) → el `merge` por
  `Zip Code` no va a perder clientes. `population` tiene **45 zip codes adicionales**
  que ningún cliente de TELCO usa (zonas de California sin clientes en este dataset),
  lo cual es perfectamente normal.

### 3.3 Distribuciones y valores

In [23]:
population[['Population']].describe()

,Population
count,"1,671.00"
mean,"20,276.38"
std,"20,689.12"
min,11.00
25%,"1,789.00"
50%,"14,239.00"
75%,"32,942.50"
max,"105,285.00"


**Distribuciones y valores — interpretación**

- **`Population`**: rango 11 - 105.285 habitantes por código postal, media ≈ 20.276,
  mediana ≈ 14.239. La distribución está sesgada a la derecha (hay zip codes muy
  poco poblados y otros muy poblados, típico en datos de población por código
  postal). El valor mínimo (11 habitantes) es bajo pero **plausible** para un código
  postal rural — no lo consideramos un error.
- **Conclusión:** tabla pequeña, limpia y sin problemas. Su única peculiaridad es que
  está a **otra granularidad** (zip code, no cliente), algo que hay que tener muy en
  cuenta al diseñar el `merge` en la ETL.

# Alerta de Granularidad en el Merge ETL (Clientes vs. Población)

Al cruzar la tabla principal con la de población, nos enfrentamos a un choque de **granularidades** (el nivel de detalle de cada fila):

* **Tabla Principal:** Granularidad fina (1 fila = 1 Cliente).
* **Tabla de Población:** Granularidad gruesa (1 fila = 1 Código Postal).

### ¿Qué ocurre al hacer el `merge`?
Dado que muchos clientes viven en el mismo código postal (relación de Muchos a Uno), **el dato de población se va a duplicar** por cada cliente que comparta *zip code*. 

**Ejemplo:** Si el CP 28001 tiene 30.000 habitantes y tenemos 3 clientes ahí, la tabla final repetirá esos 30.000 habitantes tres veces.

### El Peligro Analítico
Tras el cruce, la columna `population` se convierte en una característica del cliente. **No se puede sumar directamente** para calcular la población total de nuestras zonas, ya que la suma estaría inflada por los duplicados (ej. sumaría 90.000 en lugar de 30.000).

### Solución recomendada
* **En BI:** Mantener ambas tablas separadas relacionándolas por `zip_code` (Modelo en Estrella) en lugar de aplanarlas en una sola.
* **En SQL/Python:** Si se requiere aplanar, para calcular poblaciones totales hay que aplicar siempre un filtro de valores únicos previamente (`SELECT SUM(population) FROM (SELECT DISTINCT zip_code, population...)`).

## 4. Tabla `services`

### 4.1 Estructura básica

In [24]:
print("Shape:", services.shape)
services.head()

Shape: (7043, 30)


,Customer ID,Count,Quarter,Referred a Friend,Number of Referrals,Tenure in Months,Offer,Phone Service,Avg Monthly Long Distance Charges,Multiple Lines,Internet Service,Internet Type,Avg Monthly GB Download,Online Security,Online Backup,Device Protection Plan,Premium Tech Support,Streaming TV,Streaming Movies,Streaming Music,Unlimited Data,Contract,Paperless Billing,Payment Method,Monthly Charge,Total Charges,Total Refunds,Total Extra Data Charges,Total Long Distance Charges,Total Revenue
0,8779-QRDMV,1,Q3,No,0,1,NaN,No,0.00,No,Yes,DSL,8,No,No,Yes,No,No,Yes,No,No,Month-to-Month,Yes,Bank Withdrawal,39.65,39.65,0.00,20,0.00,59.65
1,7495-OOKFY,1,Q3,Yes,1,8,Offer E,Yes,48.85,Yes,Yes,Fiber Optic,17,No,Yes,No,No,No,No,No,Yes,Month-to-Month,Yes,Credit Card,80.65,633.30,0.00,0,390.80,"1,024.10"
2,1658-BYGOY,1,Q3,No,0,18,Offer D,Yes,11.33,Yes,Yes,Fiber Optic,52,No,No,No,No,Yes,Yes,Yes,Yes,Month-to-Month,Yes,Bank Withdrawal,95.45,"1,752.55",45.61,0,203.94,"1,910.88"
3,4598-XLKNJ,1,Q3,Yes,1,25,Offer C,Yes,19.76,No,Yes,Fiber Optic,12,No,Yes,Yes,No,Yes,Yes,No,Yes,Month-to-Month,Yes,Bank Withdrawal,98.50,"2,514.50",13.43,0,494.00,"2,995.07"
4,4846-WHAFZ,1,Q3,Yes,1,37,Offer C,Yes,6.33,Yes,Yes,Fiber Optic,14,No,No,No,No,No,No,No,Yes,Month-to-Month,Yes,Bank Withdrawal,76.50,"2,868.15",0.00,0,234.21,"3,102.36"


In [25]:
services.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 30 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Customer ID                        7043 non-null   str    
 1   Count                              7043 non-null   int64  
 2   Quarter                            7043 non-null   str    
 3   Referred a Friend                  7043 non-null   str    
 4   Number of Referrals                7043 non-null   int64  
 5   Tenure in Months                   7043 non-null   int64  
 6   Offer                              3166 non-null   str    
 7   Phone Service                      7043 non-null   str    
 8   Avg Monthly Long Distance Charges  7043 non-null   float64
 9   Multiple Lines                     7043 non-null   str    
 10  Internet Service                   7043 non-null   str    
 11  Internet Type                      5517 non-null   str    
 12  Avg

In [26]:
print("Customer ID unicos:", services['Customer ID'].nunique())
print("Customer ID duplicados:", services['Customer ID'].duplicated().sum())

Customer ID unicos: 7043
Customer ID duplicados: 0


**Estructura básica — interpretación**

- **Filas / columnas:** 7.043 filas x 30 columnas — la tabla más ancha de las 5.
- **Clave primaria:** `Customer ID`, única y sin duplicados.
- **Tipos de columna** (agrupados):
  - `Customer ID` → identificador.
  - `Count` (=1) y `Quarter` (="Q3") → **constantes**, sin información.
  - **Flags de servicios** (categóricas binarias Yes/No): `Referred a Friend`,
    `Phone Service`, `Multiple Lines`, `Internet Service`, `Online Security`,
    `Online Backup`, `Device Protection Plan`, `Premium Tech Support`,
    `Streaming TV`, `Streaming Movies`, `Streaming Music`, `Unlimited Data`,
    `Paperless Billing`.
  - **Categóricas multinivel**: `Offer` (A-E / sin oferta), `Internet Type`
    (DSL / Fiber Optic / Cable / sin internet), `Contract` (Month-to-Month / One
    Year / Two Year), `Payment Method`.
  - **Numéricas**: `Number of Referrals`, `Tenure in Months`,
    `Avg Monthly Long Distance Charges`, `Avg Monthly GB Download`,
    `Monthly Charge`, `Total Charges`, `Total Refunds`,
    `Total Extra Data Charges`, `Total Long Distance Charges`, `Total Revenue`.

### 4.2 Calidad de los datos

In [27]:
nulos = services.isnull().sum()
print("Columnas con nulos:")
print(nulos[nulos > 0])
print("\n% de nulos:")
print((nulos[nulos > 0] / len(services) * 100).round(2))

print("\nFilas duplicadas:", services.duplicated().sum())

Columnas con nulos:
Offer            3877
Internet Type    1526
dtype: int64

% de nulos:
Offer           55.05
Internet Type   21.67
dtype: float64

Filas duplicadas: 0


In [28]:
# ¿El nulo de 'Offer' significa "sin oferta activa"?
print("Offer (incluyendo nulos):")
print(services['Offer'].value_counts(dropna=False))

Offer (incluyendo nulos):
Offer
NaN        3877
Offer B     824
Offer E     805
Offer D     602
Offer A     520
Offer C     415
Name: count, dtype: int64


In [29]:
# ¿El nulo de 'Internet Type' coincide exactamente con Internet Service = 'No'?

# crosstab para ver la relación entre 'Internet Service' y si 'Internet Type' es nulo
print(pd.crosstab(services['Internet Service'], services['Internet Type'].isnull(),
                   rownames=['Internet Service'], colnames=['Internet Type es nulo']))

Internet Type es nulo  False  True 
Internet Service                   
No                         0   1526
Yes                     5517      0


**Calidad de los datos — interpretación**

- **`Offer`** tiene un **55,05% de nulos** (3.877 filas). A simple vista parece un
  problema grave, pero al investigar el significado de negocio vemos que **`NaN`
  representa "el cliente no tiene ninguna oferta promocional activa"**, no un dato
  perdido. Es información real codificada como nulo → en la ETL la imputaremos como
  categoría `'No Offer'`, **no** con media/moda.
- **`Internet Type`** tiene un **21,67% de nulos** (1.526 filas), y coincide
  **exactamente** (100%) con los 1.526 clientes que tienen `Internet Service = 'No'`.
  De nuevo, el nulo significa **"sin servicio de internet"**, no un dato perdido →
  se imputará como `'No Internet'`.
- **Duplicados:** no hay filas duplicadas.
- **Tipos de datos:** todas las columnas numéricas vienen ya como `int64`/`float64`,
  no hay importes almacenados como texto.
- **Conclusión:** esta tabla **no tiene nulos "problemáticos"** — los dos únicos
  nulos detectados son en realidad **información de negocio** (ausencia de oferta /
  ausencia de internet) que se debe imputar con una categoría explícita, no
  estadísticamente.

### 4.3 Distribuciones y valores

In [30]:
num_cols = ['Number of Referrals', 'Tenure in Months', 'Avg Monthly Long Distance Charges',
            'Avg Monthly GB Download', 'Monthly Charge', 'Total Charges', 'Total Refunds',
            'Total Extra Data Charges', 'Total Long Distance Charges', 'Total Revenue']

# describe para ver estadísticas descriptivas de las columnas numéricas
# .T para transponer el resultado y que sea más legible
services[num_cols].describe().T 

,count,mean,std,min,25%,50%,75%,max
Number of Referrals,"7,043.00",1.95,3.00,0.00,0.00,0.00,3.00,11.00
Tenure in Months,"7,043.00",32.39,24.54,1.00,9.00,29.00,55.00,72.00
Avg Monthly Long Distance Charges,"7,043.00",22.96,15.45,0.00,9.21,22.89,36.39,49.99
Avg Monthly GB Download,"7,043.00",20.52,20.42,0.00,3.00,17.00,27.00,85.00
Monthly Charge,"7,043.00",63.60,31.20,-10.00,30.40,70.05,89.75,118.75
Total Charges,"7,043.00","2,280.38","2,266.22",18.80,400.15,"1,394.55","3,786.60","8,684.80"
Total Refunds,"7,043.00",1.96,7.90,0.00,0.00,0.00,0.00,49.79
Total Extra Data Charges,"7,043.00",6.86,25.10,0.00,0.00,0.00,0.00,150.00
Total Long Distance Charges,"7,043.00",749.10,846.66,0.00,70.55,401.44,"1,191.10","3,564.72"
Total Revenue,"7,043.00","3,034.38","2,865.20",21.36,605.61,"2,108.64","4,801.15","11,979.34"


In [31]:
# Cargos mensuales negativos -> imposible, porque el cargo mensual no puede ser negativo
print("Clientes con Monthly Charge < 0:", (services['Monthly Charge'] < 0).sum())

# .loc para filtrar las filas con Monthly Charge < 0 y luego describir la columna 'Monthly Charge'
print(services.loc[services['Monthly Charge'] < 0, 'Monthly Charge'].describe())

idx_neg = services[services['Monthly Charge'] < 0].index
print("\nRango de indices con Monthly Charge < 0:", idx_neg.min(), "-", idx_neg.max())

Clientes con Monthly Charge < 0: 120
count   120.00
mean     -5.42
std       2.89
min     -10.00
25%      -8.00
50%      -5.00
75%      -3.00
max      -1.00
Name: Monthly Charge, dtype: float64

Rango de indices con Monthly Charge < 0: 6923 - 7042


In [32]:
# Total Revenue, ¿es una variable derivada de las otras 4?
calculo = (services['Total Charges'] - services['Total Refunds']
           + services['Total Extra Data Charges'] + services['Total Long Distance Charges'])
diferencia = (calculo - services['Total Revenue']).abs()

# Esto se hace para ver si hay alguna diferencia entre el cálculo y el valor de 'Total Revenue' en el dataframe
print("Diferencia maxima entre 'calculo' y 'Total Revenue':", diferencia.max())

# Esto se hace para ver cuántas filas tienen una diferencia mayor a 0.01 entre el cálculo y el valor de 'Total Revenue'
print("Filas donde no coincide (tolerancia 0.01):", (diferencia > 0.01).sum())

Diferencia maxima entre 'calculo' y 'Total Revenue': 1.8189894035458565e-12
Filas donde no coincide (tolerancia 0.01): 0


In [33]:
# Valores unicos de variables categoricas clave
for col in ['Contract', 'Payment Method', 'Internet Type', 'Offer']:
    print(f"\n{col}:")
    print(services[col].value_counts(dropna=False))


Contract:
Contract
Month-to-Month    3610
Two Year          1883
One Year          1550
Name: count, dtype: int64

Payment Method:
Payment Method
Bank Withdrawal    3909
Credit Card        2749
Mailed Check        385
Name: count, dtype: int64

Internet Type:
Internet Type
Fiber Optic    3035
DSL            1652
NaN            1526
Cable           830
Name: count, dtype: int64

Offer:
Offer
NaN        3877
Offer B     824
Offer E     805
Offer D     602
Offer A     520
Offer C     415
Name: count, dtype: int64


**Distribuciones y valores — interpretación**

- **`Tenure in Months`**: rango 1-72 (6 años), media ≈ 32,4, mediana = 29. Sin
  ceros ni valores negativos, coherente.
- **`Avg Monthly GB Download`**: rango 0-85; el 0 corresponde siempre a clientes sin
  internet (`Internet Service = 'No'`), por lo que es coherente, no un error.
- **`Monthly Charge`**: rango **-10 a 118,75**. Encontramos **120 clientes con
  `Monthly Charge` negativo** (entre -1 y -10), lo cual **es imposible** — un cargo
  mensual no puede ser negativo. Igual que en `demographics`, estos 120 registros
  están **concentrados en las últimas 120 filas del fichero** (índices 6.923-7.042),
  lo que refuerza la hipótesis de que existe un **bloque final de filas con errores
  inyectados** en varias tablas. Habrá que decidir en la ETL si se corrige (p.ej.
  `abs()`), se imputa o se descarta.
- **`Total Revenue` es una variable 100% derivada**: `Total Revenue = Total Charges -
  Total Refunds + Total Extra Data Charges + Total Long Distance Charges` se cumple
  exactamente (diferencia máxima ≈ 1.8e-12, error de redondeo) en las 7.043 filas.
  Esto es importante: **`Total Revenue` es una combinación lineal de otras 4
  columnas** → candidato a multicolinealidad perfecta en el análisis multivariante
  (PASO 3).
- **Variables categóricas multinivel**:
  - `Contract`: `Month-to-Month` (3.610), `One Year` (1.550), `Two Year` (1.883).
  - `Internet Type`: `Fiber Optic` (3.035), `DSL` (1.652), `Cable` (830), sin internet
    (1.526).
  - `Offer`: 5 ofertas (A-E) + sin oferta (3.877).
  - `Payment Method`: 4 métodos de pago.
  - No se detectan inconsistencias de mayúsculas/espacios en ninguna de ellas.
- **Conclusión:** la tabla está bien estructurada, pero tiene **dos problemas de
  calidad puntuales**: el bloque de `Monthly Charge` negativo (120 filas) y la
  redundancia exacta de `Total Revenue`.

## 5. Tabla `status`

### 5.1 Estructura básica

In [34]:
print("Shape:", status.shape)
status.head()

Shape: (7043, 8)


,Customer ID,Count,Quarter,Customer Status,Churn Label,Churn Value,Churn Category,Churn Reason
0,8779-QRDMV,1,Q3,Churned,Yes,1,Competitor,Competitor offered more data
1,7495-OOKFY,1,Q3,Churned,Yes,1,Competitor,Competitor made better offer
2,1658-BYGOY,1,Q3,Churned,Yes,1,Competitor,Competitor made better offer
3,4598-XLKNJ,1,Q3,Churned,Yes,1,Dissatisfaction,Limited range of services
4,4846-WHAFZ,1,Q3,Churned,Yes,1,Price,Extra data charges


In [35]:
status.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Customer ID      7043 non-null   str  
 1   Count            7043 non-null   int64
 2   Quarter          7043 non-null   str  
 3   Customer Status  7043 non-null   str  
 4   Churn Label      7043 non-null   str  
 5   Churn Value      7043 non-null   int64
 6   Churn Category   1869 non-null   str  
 7   Churn Reason     1869 non-null   str  
dtypes: int64(2), str(6)
memory usage: 440.3 KB


In [36]:
print("Customer ID unicos:", status['Customer ID'].nunique())
print("Customer ID duplicados:", status['Customer ID'].duplicated().sum())

Customer ID unicos: 7043
Customer ID duplicados: 0


**Estructura básica — interpretación**

- **Filas / columnas:** 7.043 filas x 8 columnas.
- **Clave primaria:** `Customer ID`, única y sin duplicados.
- **Tipos de columna:**
  - `Customer ID` → identificador.
  - `Count` (=1) y `Quarter` (="Q3") → constantes, igual que en `services`.
  - `Customer Status` → categórica (`Stayed` / `Churned` / `Joined`).
  - `Churn Label` → categórica binaria (`Yes` / `No`).
  - `Churn Value` → numérica binaria (0 / 1) — **es la variable objetivo**.
  - `Churn Category` → categórica multinivel (solo para clientes que han hecho churn).
  - `Churn Reason` → categórica de muy alta cardinalidad (motivo detallado del
    churn, solo para clientes que han hecho churn).

### 5.2 Calidad de los datos

In [37]:
nulos = status.isnull().sum()
print("Columnas con nulos:")
print(nulos[nulos > 0])
print("\n% de nulos:")
print((nulos[nulos > 0] / len(status) * 100).round(2))

print("\nFilas duplicadas:", status.duplicated().sum())

Columnas con nulos:
Churn Category    5174
Churn Reason      5174
dtype: int64

% de nulos:
Churn Category   73.46
Churn Reason     73.46
dtype: float64

Filas duplicadas: 0


In [38]:
# Relacion entre Customer Status, Churn Label y Churn Value para ver si hay alguna incoherencia
status.groupby(['Customer Status', 'Churn Label'])['Churn Value'].agg(['min', 'max', 'count'])

,,min,max,count
Customer Status,Churn Label,,,
Churned,Yes,1,1,1869
Joined,No,0,0,454
Stayed,No,0,0,4720


In [39]:
# ¿Los nulos de Churn Category/Reason corresponden solo a clientes que NO han hecho churn?
# .apply(lambda x: x.isnull().sum()) para contar los nulos en cada grupo
status.groupby('Customer Status')['Churn Category'].apply(lambda x: x.isnull().sum())

Customer Status
Churned       0
Joined      454
Stayed     4720
Name: Churn Category, dtype: int64

**Calidad de los datos — interpretación**

- **`Churn Category`** y **`Churn Reason`** tienen ambas un **73,46% de nulos**
  (5.174 filas). Al comprobarlo, los nulos corresponden **exactamente** a los 5.174
  clientes con `Customer Status` = `Stayed` o `Joined` (es decir, clientes que **no**
  han hecho churn). Para los 1.869 clientes `Churned`, ambas columnas están siempre
  rellenas. **No es un problema de recogida**: simplemente no existe "motivo de
  churn" para quien no se ha ido. Se imputará como categoría `'No Churn'` /
  `'Sin motivo'`.
- **Duplicados:** ninguno.
- **Tipos de datos:** correctos.
- **Coherencia interna del target:**
  - `Customer Status = 'Churned'` ⟺ `Churn Label = 'Yes'` ⟺ `Churn Value = 1`
    (1.869 filas, 100% consistente).
  - `Customer Status ∈ {'Stayed', 'Joined'}` ⟺ `Churn Label = 'No'` ⟺
    `Churn Value = 0` (5.174 filas).
  - Es decir, `Churn Label` y `Churn Value` son **la misma información en dos
    formatos** (texto vs binario), y `Customer Status` añade un matiz extra: separa
    a los clientes activos en `Stayed` (4.720, clientes "antiguos" que se han
    quedado) y `Joined` (454, clientes nuevos de esta misma cuarter, con
    `Tenure in Months` entre 1 y 3).

### 5.3 Distribuciones y valores

In [40]:
print("Customer Status:")
print(status['Customer Status'].value_counts())

print("\nChurn Value (variable objetivo):")
print(status['Churn Value'].value_counts())
print("Tasa de churn global: {:.2%}".format(status['Churn Value'].mean()))

Customer Status:
Customer Status
Stayed     4720
Churned    1869
Joined      454
Name: count, dtype: int64

Churn Value (variable objetivo):
Churn Value
0    5174
1    1869
Name: count, dtype: int64
Tasa de churn global: 26.54%


In [41]:
print("Churn Category (clientes que han hecho churn):")
print(status['Churn Category'].value_counts(dropna=False))

print("\nTop 10 Churn Reason:")
print(status['Churn Reason'].value_counts(dropna=False).head(10))

Churn Category (clientes que han hecho churn):
Churn Category
NaN                5174
Competitor          841
Attitude            314
Dissatisfaction     303
Price               211
Other               200
Name: count, dtype: int64

Top 10 Churn Reason:
Churn Reason
NaN                                          5174
Competitor had better devices                 313
Competitor made better offer                  311
Attitude of support person                    220
Don't know                                    130
Competitor offered more data                  117
Competitor offered higher download speeds     100
Attitude of service provider                   94
Price too high                                 78
Product dissatisfaction                        77
Name: count, dtype: int64


**Distribuciones y valores — interpretación**

- **`Churn Value`** es binaria (0/1), sin valores fuera de rango. **Tasa de churn
  global ≈ 26,5%** (1.869 de 7.043 clientes) — un **desbalance moderado** (≈ 1:2,8),
  algo a tener muy en cuenta al dividir train/test y elegir métricas en el PASO 4.
- **`Churn Category`** (solo para `Churned`): `Competitor` (841), `Attitude` (314),
  `Dissatisfaction` (303), `Price` (211), `Other` (200). El motivo más común de
  abandono es la **competencia**, no el precio — un primer indicio relevante para
  las hipótesis de Sebastián.
- **`Churn Reason`** tiene alta cardinalidad (decenas de motivos detallados); los
  más frecuentes son `Competitor had better devices` (313) y
  `Competitor made better offer` (311), coherentes con `Churn Category = 'Competitor'`.
- **Conclusión:** tabla limpia y coherente. Es la tabla que contiene **la variable
  objetivo** del proyecto.

## 6. Primer vistazo: variables potencialmente relacionadas con el churn

Antes de responder a las preguntas de reflexión, hacemos un cruce rápido —solo a
nivel exploratorio, sin limpiar ni integrar todavía— entre `Churn Value` (de
`status`) y algunas variables candidatas de `services` y `location`, para tener
intuiciones de cara al PASO 3 (análisis multivariante).

In [42]:
# Merge minimo solo para explorar (no es la ETL definitiva)
exploracion = (
    services[['Customer ID', 'Contract', 'Internet Type', 'Offer', 'Tenure in Months', 'Monthly Charge']]
    .merge(status[['Customer ID', 'Churn Value']], on='Customer ID')
    .merge(location[['Customer ID', 'City']], on='Customer ID')
)

print("Tasa de churn por tipo de contrato:")
print(exploracion.groupby('Contract')['Churn Value'].agg(['mean', 'count']))

Tasa de churn por tipo de contrato:
                mean  count
Contract                   
Month-to-Month  0.46   3610
One Year        0.11   1550
Two Year        0.03   1883


In [43]:
print("Tasa de churn: San Diego vs resto de California:")
exploracion['es_san_diego'] = exploracion['City'] == 'San Diego'
print(exploracion.groupby('es_san_diego')['Churn Value'].agg(['mean', 'count']))

Tasa de churn: San Diego vs resto de California:
              mean  count
es_san_diego             
False         0.25   6758
True          0.65    285


In [44]:
print("Tasa de churn por tipo de Internet:")
print(exploracion.groupby('Internet Type', dropna=False)['Churn Value'].agg(['mean', 'count']))

print("\nTasa de churn por Offer:")
print(exploracion.groupby('Offer', dropna=False)['Churn Value'].agg(['mean', 'count']))

# .corr para ver la correlación entre Tenure in Months, Monthly Charge y Churn Value
print("\nCorrelacion simple Tenure / Monthly Charge con Churn Value:")
print(exploracion[['Tenure in Months', 'Monthly Charge', 'Churn Value']].corr()['Churn Value'])

Tasa de churn por tipo de Internet:
               mean  count
Internet Type             
Cable          0.26    830
DSL            0.19   1652
Fiber Optic    0.41   3035
NaN            0.07   1526

Tasa de churn por Offer:
         mean  count
Offer               
Offer A  0.07    520
Offer B  0.12    824
Offer C  0.23    415
Offer D  0.27    602
Offer E  0.53    805
NaN      0.27   3877

Correlacion simple Tenure / Monthly Charge con Churn Value:
Tenure in Months   -0.35
Monthly Charge      0.19
Churn Value         1.00
Name: Churn Value, dtype: float64


**Lectura rápida de estos primeros cruces** (se profundizará en el PASO 3):

- **Contrato Month-to-Month**: tasa de churn ≈ **45,8%**, frente a 10,7% (One Year) y
  2,5% (Two Year). Diferencia muy grande → primera señal a favor de la hipótesis 1 de
  Sebastián.
- **San Diego**: tasa de churn ≈ **64,9%**, frente a 24,9% en el resto de California.
  Diferencia enorme → primera señal a favor de la hipótesis 2 de Sebastián.
- **Fiber Optic**: tasa de churn ≈ 40,7%, claramente por encima de DSL (18,6%) y de
  "sin internet" (7,4%).
- **`Offer E`**: tasa de churn ≈ 52,9%, la más alta de todas las ofertas.
- **`Tenure in Months`**: correlación negativa moderada con el churn (≈ -0,35) — a
  menor antigüedad, mayor probabilidad de churn.
- **`Monthly Charge`**: correlación positiva débil (≈ 0,19).

Estas intuiciones se confirmarán o descartarán formalmente con Cramér's V y
correlación de Pearson en el PASO 3.

## Preguntas de reflexión — EDA

### 1. ¿Qué tabla contiene la variable objetivo? ¿Cómo está codificada?

La variable objetivo está en la tabla **`status`**, en la columna **`Churn Value`**,
codificada como **numérica binaria** (`0` = el cliente se queda, `1` = el cliente ha
hecho churn). Existe además una versión textual redundante, `Churn Label`
(`'No'` / `'Yes'`), que es exactamente la misma información en otro formato — para
el modelado usaremos `Churn Value` por ser numérica.

La tasa de churn global es **≈ 26,5%** (1.869 de 7.043 clientes), un desbalance
moderado que tendremos en cuenta en el PASO 4 (estratificación del split y elección
de métricas).

### 2. ¿Cuántos clientes distintos hay en el dataset? ¿Coincide ese número en todas las tablas?

Hay **7.043 clientes distintos** (`Customer ID` único), y este número **coincide
exactamente** en `demographics`, `location`, `services` y `status` — las cuatro
tablas comparten el mismo conjunto de 7.043 `Customer ID`, sin ninguno "de más" o
"de menos" en ninguna de ellas.

La **única excepción es `population`**, que **no está a nivel de cliente**: tiene
1.671 filas, una por cada **código postal** (`Zip Code`) distinto presente en
California. De los 1.626 zip codes que usan los clientes de TELCO (según
`location`), **los 1.626 existen en `population`** (0 sin match), por lo que el
`merge` por `Zip Code` no perderá información de población.

### 3. ¿Qué columnas tienen nulos que podrían desembocar en una reunión e interpretación con negocio?

Encontramos nulos en 4 columnas, pero los 4 casos tienen una explicación de negocio
clara y **no requieren, en principio, una reunión** porque el propio patrón de los
datos confirma el significado:

- `services.Offer` (55,05% nulos) → cliente sin oferta promocional activa
  (`'No Offer'`).
- `services.Internet Type` (21,67% nulos) → coincide al 100% con
  `Internet Service = 'No'` (`'No Internet'`).
- `status.Churn Category` y `status.Churn Reason` (73,46% nulos cada una) →
  coinciden al 100% con clientes que no han hecho churn (`'No Churn'`).

Dicho esto, **sí dejaríamos constancia para negocio** de dos puntos relacionados con
estas columnas, aunque no sean "nulos problemáticos":

- `Churn Category` solo tiene 5 categorías (`Competitor`, `Attitude`,
  `Dissatisfaction`, `Price`, `Other`) — convendría confirmar con Sebastián/Amalia
  si esta taxonomía es la definitiva o si "Other" (200 casos) esconde motivos que
  el negocio querría desglosar.
- La columna `Offer` solo está rellena para clientes con `Internet Service`
  contratado en algunos casos — convendría confirmar si las ofertas A-E aplican
  también a clientes sin internet o es casuística.

### 4. ¿Encontraste algún valor que te parezca un error o una inconsistencia? ¿Cómo lo tratarías?

Sí, encontramos **tres inconsistencias claras**, dos de ellas concentradas en un
mismo "bloque final" de filas que parece contener errores inyectados deliberadamente:

1. **`demographics.Gender`** mezcla dos codificaciones: `Male`/`Female` (5.768
   filas) y `M`/`F` (1.275 filas). **Tratamiento**: normalizar todo a una única
   convención (p.ej. `Male`/`Female`) en la ETL — es un simple mapeo de texto, sin
   pérdida de información.

2. **`demographics.Age` > 100** en 109 clientes (rango 101-119), todos en las
   últimas ~110 filas del fichero, y además **inconsistentes con sus propios flags**
   (89 de ellos tienen `Senior Citizen = 'No'` pese a tener `Age >= 65`; 20 tienen
   `Under 30 = 'Yes'` pese a tener `Age >= 30`). **Tratamiento**: dado que afecta a
   ~1,5% de los clientes y las edades son fisiológicamente poco plausibles
   combinadas con flags contradictorios, lo trataríamos como un **bloque de error de
   carga**. Opciones a decidir en la ETL: (a) recalcular `Under 30`/`Senior Citizen`
   a partir de `Age` para todo el dataset (resuelve la inconsistencia, pero no la
   edad anómala en sí), o (b) tratar `Age > 100` como valor faltante e imputar. Nos
   inclinamos por (a) + marcar estas filas para revisión, ya que eliminarlas
   perderíamos el resto de información (servicios, contrato, etc.) de esos
   clientes.

3. **`services.Monthly Charge` < 0** en 120 clientes (rango -1 a -10), también
   concentrados en las últimas 120 filas del fichero. Un cargo mensual **no puede**
   ser negativo. **Tratamiento**: lo más razonable es tomar el **valor absoluto**
   (`abs()`), ya que la magnitud parece correcta y solo el signo está mal — pero
   se documentará la decisión y se podría validar contra `Total Charges` /
   `Tenure in Months` si se quisiera ser más estrictos.

### 5. ¿Hay columnas que parecen redundantes o que contengan la misma información de distintas formas?

Sí, varias:

- **`Count`** (=1 siempre) en `demographics`, `location`, `services` y `status`, y
  **`Quarter`** (="Q3" siempre) en `services` y `status` → constantes sin
  información, candidatas a eliminar.
- **`location.Country`** (="United States") y **`location.State`** (="California")
  → constantes (todo el dataset es del mismo país/estado), sin varianza útil para
  el modelo, aunque confirman el alcance geográfico.
- **`population.ID`** → es simplemente la posición de la fila + 1, sin información
  propia.
- **`status.Churn Label`** (`Yes`/`No`) y **`status.Churn Value`** (`1`/`0`) son
  **la misma variable en dos formatos**.
- **`demographics.Under 30`** y **`demographics.Senior Citizen`** son, salvo en el
  bloque de errores ya descrito, **flags derivados de `Age`** (no aportan
  información adicional a la edad numérica, aunque pueden ser útiles como
  categorías directas).
- **`services.Total Revenue`** es una **combinación lineal exacta** de
  `Total Charges`, `Total Refunds`, `Total Extra Data Charges` y
  `Total Long Distance Charges` (`Total Revenue = Total Charges - Total Refunds +
  Total Extra Data Charges + Total Long Distance Charges`, se cumple en el 100% de
  las filas). Es la redundancia más relevante de cara al PASO 3 (VIF /
  multicolinealidad).
- **`demographics.Number of Dependents`** y **`demographics.Dependents`**: el flag
  `Dependents` es 100% derivable de `Number of Dependents` (`'Yes'` ⟺
  `Number of Dependents > 0`).

### 6. ¿Qué variables crees que podrían tener más relación con el churn, solo con este primer vistazo?

Según los cruces exploratorios de la sección 6:

- **`Contract`** (tipo de contrato): diferencia enorme entre Month-to-Month (45,8%
  churn) y Two Year (2,5% churn). Es, a simple vista, la variable categórica con
  mayor relación aparente con el churn — y **confirma la primera hipótesis de
  Sebastián**.
- **`City` = San Diego**: tasa de churn del 64,9% frente al 24,9% del resto de
  California — la diferencia más extrema de todas las que hemos visto, y **confirma
  la segunda hipótesis de Sebastián**.
- **`Internet Type`**: Fiber Optic destaca con un 40,7% de churn frente al 18,6% de
  DSL y el 7,4% de quienes no tienen internet.
- **`Offer`**: la oferta `Offer E` tiene una tasa de churn del 52,9%, muy por encima
  del resto.
- **`Tenure in Months`**: correlación negativa (-0,35) con el churn — los clientes
  más nuevos abandonan más.
- **`Monthly Charge`**: correlación positiva débil (0,19) — pagar más se asocia
  ligeramente con más churn, aunque mucho menos que las anteriores.

Todo esto son **intuiciones de primer vistazo**: en el PASO 3 las confirmaremos de
forma rigurosa con Cramér's V (categóricas) y correlación de Pearson / VIF
(numéricas).